In [ ]:
!pip install langdetect

In [ ]:
!pip install googletrans==4.0.0-rc1

In [1]:
import speech_recognition as sr
from googletrans import Translator
from gtts import gTTS
import os
import time
import pygame
import openai
from langdetect import detect

# Initialize pygame for audio output
pygame.mixer.init()

# Initialize the speech recognition
recognizer = sr.Recognizer()

# Function to capture voice command through microphone
def take_command():
    with sr.Microphone() as source:
        print("Listening...")
        recognizer.adjust_for_ambient_noise(source)
        try:
            audio = recognizer.listen(source, timeout=10, phrase_time_limit=10)
        except sr.WaitTimeoutError:
            print("Timeout reached. No speech detected.")
            return None, None

    try:
        print("Recognizing...")
        result = recognizer.recognize_google(audio, show_all=True)
        if 'alternative' in result and result['alternative']:
            query = result['alternative'][0]['transcript']
            lang = detect(query)  # Detect language using langdetect
            print(f"User said: {query} in {lang}\n")
            return query, lang
        else:
            print("No speech understood, please try again.")
            return None, None
    except sr.UnknownValueError:
        print("Google Speech Recognition could not understand the audio")
    except sr.RequestError as e:
        print(f"Could not request results from Google Speech Recognition service; {e}")
    return None, None

# Main interaction loop
query, input_lang = take_command()
while query is None:
    query, input_lang = take_command()

if query:  # Proceed only if a valid query is captured
    # Translate the user's command to English
    translator = Translator()
    translated_text = translator.translate(query, dest='en').text

    # API call to OpenAI ChatGPT
    openai.api_key = os.getenv('OPENAI_API_KEY')
    response = openai.ChatCompletion.create(
        model="gpt-4-turbo-preview",
        messages=[
            {"role": "system", "content": "Please provide helpful and accurate information."},
            {"role": "user", "content": translated_text}
        ],
        temperature=0.5,
        max_tokens=150
    )
    gpt_response = response['choices'][0]['message']['content']

    # Translate the ChatGPT response back to the original input language
    translated_response = translator.translate(gpt_response, dest=input_lang).text

    # Convert translated response to speech and save it
    speech = gTTS(text=translated_response, lang=input_lang, slow=False)
    speech.save("response_voice.mp3")

    # Play the translated speech using pygame
    response_sound = pygame.mixer.Sound("response_voice.mp3")
    response_sound.play()

    # Wait until the audio finishes playing
    while pygame.mixer.get_busy():
        time.sleep(1)

    # Delete the temporary file
    os.remove('response_voice.mp3')

    # Print translated text
    print("GPT Response in Original Language:", translated_response)


pygame 2.5.2 (SDL 2.28.3, Python 3.11.4)
Hello from the pygame community. https://www.pygame.org/contribute.html
Listening...
Recognizing...
User said: hello who is this in en

GPT Response in Original Language: Hello! I'm an AI developed by OpenAI, designed to provide information and answer questions to the best of my ability based on the knowledge I've been trained on. How can I assist you today?
